# CALPHAD workflow with ATAT and machine-learned potentials

This tutorial fits a CALPHAD database for Ni-Re with the `CalphadMaker`
workflow and draws the phase diagram. The energies come from four
machine-learned potentials, so no DFT is needed. The workflow is described in
the [force field documentation](https://materialsproject.github.io/atomate2/user/codes/forcefields.html#calphad).

This workflow is new and has not been tested widely. It might still change in
future versions.

## Recommendations

These settings worked for Ni-Re with all four potentials.

- Use the default liquid cell of 864 atoms. With 256 atoms the liquid mixing
  energy of one potential was off by 2 kJ/mol.
- Set `melt_temperature` high enough that every composition melts. We used
  4500 K.
- Run all compositions at one `liquid_temperature`, as low as possible while
  every composition stays liquid. We used 2800 K.
- Check that `relaxation_strain` of each solid is below 0.1 and that
  `mean_squared_displacement` of each liquid is well above 1 Å².
- Run the liquids with a few random seeds and compare the diagrams. The
  scatter between runs moved the lowest temperature of the liquid by up to
  85 K.

## How the workflow works

The workflow follows the `sqs2tdb` approach of ATAT (van de Walle et al.,
Calphad 58, 70 (2017)). It has four steps.

1. `sqs2tdb` copies the special quasirandom structures (SQS) of each lattice
   from the ATAT database. An SQS is a small cell whose random occupation of
   the sites mimics a random alloy. At level 2 these are the pure elements and
   the compositions with 25, 50 and 75 at% of each element.
2. Each solid SQS is relaxed with the force field, including the cell.
3. Each liquid SQS is repeated three times along each lattice vector, which
   gives 864 atoms. It is melted at `melt_temperature` and then run at
   `liquid_temperature`. Both runs are NPT at zero pressure. The mean potential
   energy of the second run is the energy of the liquid.
4. `sqs2tdb` fits the mixing energies of each lattice and writes one TDB file.
   A TDB file is a CALPHAD database that pycalphad can read.

The mixing energy of a phase is a Redlich-Kister sum,
x<sub>Ni</sub> x<sub>Re</sub> [L0 + L1 (x<sub>Ni</sub> − x<sub>Re</sub>) + ...].
With the terms used here only L0 is fitted. For lattices in the SGTE database,
`sqs2tdb` takes the free energies of the pure elements from SGTE (Dinsdale,
Calphad 15, 317 (1991)). The melting points of pure Ni and Re are therefore
the experimental ones.

## Installation

The `ase` extra is needed, plus MACE and pycalphad:

```
pip install 'atomate2[ase]' 'mace-torch>=0.3.16' pycalphad
```

GRACE needs `tensorpotential`, which brings TensorFlow. PyTorch and
TensorFlow each install their own CUDA libraries, so we ran GRACE in a separate
environment:

```
pip install 'atomate2[ase]' tensorpotential pycalphad
export TF_USE_LEGACY_KERAS=1
```

GRACE saves its models in `~/.cache/grace`. Set `GRACE_CACHE` to save them
somewhere else.

ATAT is not a Python package. Besides the `sqs2tdb` script, the workflow needs
three ATAT programs, which build in a few seconds:

```
curl -sSLO https://axelvandewalle.github.io/www-avdw/atat/atat3_52.tar.gz
tar xzf atat3_52.tar.gz
make -C atat/src cellcvrt nntouch lsfit
echo "set atatdir=$PWD/atat" > ~/.atat.rc
export PATH=$PWD/atat/src:$PATH
```

`sqs2tdb` calls the other programs by name, so the ATAT `src` folder must be on
the `PATH` of the shell that starts Jupyter. The next cell checks this.

In [ ]:
import shutil

for program in ("sqs2tdb", "cellcvrt", "nntouch", "lsfit"):
    if shutil.which(program) is None:
        raise OSError(f"{program} is not on the PATH")

## Setting up the workflow

### The potentials

The tutorial uses four foundation models.

- **MACE-OMAT-0-medium.** `mace_mp(model="medium-omat-0")` downloads it once and
  caches it under `~/.cache/mace`.
- **MACE-MATPES-PBE-0** and **MACE-MATPES-r2SCAN-0.** These are MACE-OMAT-0
  fine-tuned on the MatPES data set at the PBE and r2SCAN levels. Download the
  two files from the `mace_matpes_0` release of
  [mace-foundations](https://github.com/ACEsuit/mace-foundations/releases/tag/mace_matpes_0):
  - [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  - [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model)
- **GRACE-2L-OMAT.** A two-layer GRACE model fitted on OMat24. `grace_fm` of
  `tensorpotential` downloads it once by name. atomate2 has no name for GRACE,
  so the calculator is given as a dictionary that points to `grace_fm`.

Set the paths to the two MACE files in the next cell. The liquid MD needs a
GPU, so MACE runs on `"cuda"`. The mixing energies are small differences of
large total energies, so MACE runs in `float64`. GRACE runs in `float64` by
default.

In [ ]:
MACE = {"device": "cuda", "default_dtype": "float64"}
GRACE = {
    "@module": "tensorpotential.calculator.foundation_models",
    "@callable": "grace_fm",
}

# Replace the two paths with the files you downloaded.
FORCE_FIELDS = {
    "MACE-OMAT-0-medium": ("MACE-MP-0", {"model": "medium-omat-0", **MACE}),
    "MACE-MATPES-PBE-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-pbe-omat-ft.model", **MACE},
    ),
    "MACE-MATPES-r2SCAN-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-r2scan-omat-ft.model", **MACE},
    ),
    "GRACE-2L-OMAT": (GRACE, {"model": "GRACE-2L-OMAT"}),
}

### Lattices and terms

Ni is stable in FCC_A1 and Re in HCP_A3. We fit these two lattices and the
liquid. We also fit two ordered lattices of the ATAT database, NI3SN_D019 and
NI4MO_D1A. They test whether a force field makes an ordered phase stable.

`terms` gives the lines of the `sqs2tdb` `terms.in` file. Each line is an
`order,level` pair. Order 1 gives the end members and order 2 the mixing.
Level 0 fits L0 only. The ordered lattices have two sublattices, so their lines
have one pair per sublattice. `sqs2tdb` applies each line to every
sublattice, so `2,0:1,0` fits mixing on each sublattice while the other one
is pure.

### Liquid temperatures

The liquid temperatures depend on the system, so they have no defaults. The
melt at 4500 K makes every composition melt within the 10 ps of the first run.
All compositions are then run at 2800 K. This is below the melting point of Re
(3458 K in SGTE). The melted cells still stay liquid during the 20 ps run. The
section on the liquid settings explains this choice.

In [ ]:
from atomate2.forcefields.flows.calphad import CalphadMaker

TERMS = {
    "FCC_A1": ["1,0", "2,0"],
    "HCP_A3": ["1,0", "2,0"],
    "NI3SN_D019": ["1,0:1,0", "2,0:1,0"],
    "NI4MO_D1A": ["1,0:1,0", "2,0:1,0"],
    "LIQUID": ["1,0", "2,0"],
}


def get_maker(model: str) -> CalphadMaker:
    """Return the CALPHAD maker for Ni-Re with one of the potentials."""
    force_field, calculator_kwargs = FORCE_FIELDS[model]
    return CalphadMaker.from_force_field_name(
        force_field,
        melt_temperature=4500,
        liquid_temperature=2800,
        calculator_kwargs=calculator_kwargs,
        lattices=list(TERMS),
        terms=TERMS,
    )


flow = get_maker("MACE-OMAT-0-medium").make(["Ni", "Re"])
[job.name for job in flow.jobs]

## Running the workflow

The flow computes 29 SQS, 24 solids and 5 liquids. On one A100 GPU the solid
relaxations took a few minutes together. Each liquid run took 40 to 46 minutes
with MACE and about 22 minutes with GRACE-2L-OMAT. `run_locally` runs the
liquids one after the other, which takes about 3.5 hours with MACE. With a
workflow manager the five liquids can run at the same time on five GPUs. The
next cell is therefore off by default.

`create_folders=True` gives each job its own folder. The fit job runs
`sqs2tdb` in its folder.

In [ ]:
from jobflow import run_locally

RUN_WORKFLOW = False
if RUN_WORKFLOW:
    responses = run_locally(flow, create_folders=True, ensure_success=True)
    calphad_doc = responses[flow.output.uuid][1].output

## Checking the output

The output is a `CalphadDoc`. It holds the TDB file in `tdb` and the energy of
each SQS in `calculations`, together with three checks.

- `relaxation_strain` of a solid measures how far the relaxed cell is from the
  ideal lattice. The ATAT `checkrelax` help calls values above 0.1 too large.
- `is_force_converged` says whether the relaxation reached `fmax` within its
  steps.
- `mean_squared_displacement` of a liquid keeps growing during the run, because
  the atoms keep moving. In a crystal it stays well below 1 Å².

In [ ]:
import pandas as pd

checks = None
if RUN_WORKFLOW:
    checks = pd.DataFrame(
        [
            calc.model_dump(exclude={"structure", "dir_name"})
            for calc in calphad_doc.calculations
        ]
    )
checks

In our runs the largest strain was 0.034 with MACE-OMAT-0-medium and 0.038 with
GRACE-2L-OMAT. A few relaxations did not reach `fmax` within the default 500
steps. With 2000 steps the MACE energies did not change. We did not repeat the
GRACE relaxations. At 2800 K the mean squared displacement was 14 to 17 Å² for
pure Re and 109 to 122 Å² for pure Ni with all four potentials.

## Fitting the TDB and plotting

To draw the diagrams without running the workflow, we refit the TDB files from
the energies of our runs. The file holds the total energy in eV of each SQS
cell. The liquid energies are per 32-atom SQS, from the first of three MD runs
at 2800 K.

`fit_tdb` is the last job of the flow, and `.original` calls its function
directly. It writes the `sqs2tdb` folders into the current folder, so each fit
runs in a folder of its own.

In [ ]:
import contextlib
import json
from pathlib import Path

from mock_vasp import TEST_DIR

from atomate2.common.jobs.calphad import fit_tdb

energies = json.loads((TEST_DIR / "common/calphad/ni_re_energies.json").read_text())
tdbs = {}
for model, calculations in energies.items():
    fit_dir = Path("calphad_fits", model)
    fit_dir.mkdir(parents=True, exist_ok=True)
    with contextlib.chdir(fit_dir):
        tdbs[model] = fit_tdb.original(["Ni", "Re"], 2, TERMS, calculations).tdb

The next cell draws the four diagrams. The dotted line is the experimental
peritectic at 1834 K.

In [ ]:
import matplotlib.pyplot as plt
from pycalphad import Database, binplot
from pycalphad import variables as v

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
for ax, (model, tdb) in zip(axes.flat, tdbs.items(), strict=True):
    db = Database.from_string(tdb, fmt="tdb")
    binplot(
        db,
        ["NI", "RE"],
        sorted(db.phases),
        {v.X("RE"): (0, 1, 0.02), v.T: (1000, 4000, 25), v.P: 101325, v.N: 1},
        plot_kwargs={"ax": ax},
    )
    ax.axhline(1834, color="black", ls=":")
    ax.set_title(model)
    ax.label_outer()
for ax in axes.flat[:-1]:
    ax.get_legend().remove()
fig.tight_layout()
plt.show()

The diagram is decided mostly by L0 of the liquid and of FCC_A1. The next cell
reads L0 in J/mol from each TDB file.

In [ ]:
def get_l0(tdb: str, lattice: str) -> float:
    """Return L0 in J/mol of a lattice from a TDB file."""
    line = next(line for line in tdb.splitlines() if f"L({lattice},NI,RE;0)" in line)
    return float(line.split()[3])


lattices = ("FCC_A1", "HCP_A3", "LIQUID")
l0 = {
    model: [get_l0(tdb, lattice) for lattice in lattices] for model, tdb in tdbs.items()
}
pd.DataFrame(l0, index=lattices).T.round()

## Ni-Re results

### Phase diagrams

The four potentials give different diagrams on the Ni side.

- **MACE-OMAT-0-medium.** The liquid and FCC_A1 meet at a minimum at about
  1710 K, between 10 and 25 at% Re.
- **MACE-MATPES-PBE-0 and MACE-MATPES-r2SCAN-0.** The liquid is stable far
  below the melting point of Ni. It ends in a eutectic between NI4MO_D1A and
  HCP_A3 at about 33 at% Re, at 1570 K (PBE) and 1450 K (r2SCAN).
- **GRACE-2L-OMAT.** At about 1790 K the liquid and HCP_A3 form NI4MO_D1A in a
  peritectic. Near Ni a eutectic at 1730 K joins the liquid, FCC_A1 and
  NI4MO_D1A.

With the two MatPES potentials, L0 of the liquid is much more negative than L0
of FCC_A1. Adding Re to Ni therefore stabilizes the liquid and lowers the
liquidus. With MACE-OMAT-0-medium and GRACE-2L-OMAT the two are closer, so the
liquidus stays near the melting point of Ni.

### Comparison with experiment

In experiment Ni-Re has a peritectic L + HCP → FCC at 1834 K (1561 °C). The
liquid at the peritectic has about 11 at% Re, FCC 20 at% and HCP 60 at%
(Boettinger et al., Metall. Mater. Trans. A (2019)). No ordered phase has
been reported in Ni-Re as far as we know.

GRACE-2L-OMAT comes closest. Its peritectic is about 40 K below the
experimental one. It forms the ordered NI4MO_D1A phase in place of FCC_A1.

The lowest temperature at which the liquid is stable, for each of the three MD
runs at 2800 K, is in the next table. It is in K, from `pycalphad.equilibrium`
on a grid with steps of 0.01 in composition and 5 K in temperature.

| Potential | Run 1 | Run 2 | Run 3 |
|---|---|---|---|
| MACE-OMAT-0-medium | 1710 | 1705 | 1695 |
| MACE-MATPES-PBE-0 | 1570 | 1655 | 1595 |
| MACE-MATPES-r2SCAN-0 | 1450 | 1435 | 1460 |
| GRACE-2L-OMAT | 1730 | 1725 | 1715 |

All four potentials make ordered phases stable. NI4MO_D1A is stable up to
1160 K (MACE-OMAT-0-medium), 1660 K (PBE), 1460 K (r2SCAN) and 1790 K
(GRACE-2L-OMAT). With GRACE-2L-OMAT, NI3SN_D019 forms from HCP_A3 near 75 at%
Re below 1910 K.

### Comparison with PhaseForge

Zhu et al. (npj Comput. Mater. 11, 340 (2025)) computed Ni-Re with the same
`sqs2tdb` approach in their PhaseForge code and with GRACE-2L-OMAT. Their
peritectic is at 1904 K (1631 °C), 70 K above the experimental one. We ran
PhaseForge with GRACE-2L-OMAT and its default settings and got their diagram.
Its solid mixing terms agree with ours to 0.1 J/mol. Its L0 of the liquid is
+3284 J/mol, against −3860 J/mol here.

PhaseForge runs each 32-atom liquid SQS for 2 ps in NVT, at 50 K above the
mean melting point of its elements. In our PhaseForge run the mean squared
displacement of all five cells was 0.05 to 0.11 Å². The cells stayed
crystalline, so their L0 of the liquid is a solid mixing energy. Zhu et al.
note that their agreement with experiment may come from a cancellation of
errors.

## Choosing the liquid settings

### Size of the liquid cell

We ran each liquid SQS with three random seeds, set by `mb_velocity_seed` of
both liquid makers. These runs used 3600 K, not the 2800 K used above. The
next table gives the mean and standard deviation of L0 of the liquid in J/mol.
We ran GRACE-2L-OMAT only at 864 atoms.

| Potential | 256 atoms | 864 atoms | 2048 atoms |
|---|---|---|---|
| MACE-OMAT-0-medium | −1347 ± 593 | −3381 ± 449 | −3098 ± 103 |
| MACE-MATPES-PBE-0 | −17720 ± 1897 | −17141 ± 859 | −16867 ± 951 |
| MACE-MATPES-r2SCAN-0 | −14700 ± 1149 | −15741 ± 1117 | −15250 ± 631 |
| GRACE-2L-OMAT | | −7320 ± 1736 | |

From 864 to 2048 atoms the mean changes by 270 to 490 J/mol, within the
scatter between runs. From 256 to 864 atoms it changes by 2030 J/mol with
MACE-OMAT-0-medium. So 256 atoms is too small, and the workflow uses 864
atoms by default. We did not check the size at 2800 K. On one A100 GPU a MACE
liquid run took about 15 minutes at 256 atoms and 100 minutes at 2048 atoms.

### Temperature of the liquid MD

The liquid L0 comes from MD at `liquid_temperature`, so it depends on that
choice. The next table gives L0 of the liquid in J/mol at 864 atoms, as the mean
and standard deviation of three runs.

| Potential | 2800 K | 3200 K | 3600 K | 4000 K |
|---|---|---|---|---|
| MACE-OMAT-0-medium | +65 ± 414 | | −3381 ± 449 | |
| MACE-MATPES-PBE-0 | −16125 ± 1706 | | −17141 ± 859 | |
| MACE-MATPES-r2SCAN-0 | −14000 ± 500 | | −15741 ± 1117 | |
| GRACE-2L-OMAT | −4693 ± 1156 | −5989 ± 652 | −7320 ± 1736 | −9137 ± 1617 |

With GRACE-2L-OMAT, L0 changes by −3.7 ± 0.8 J/mol/K. With the MACE potentials
it rises by 1.0 to 3.4 kJ/mol from 3600 to 2800 K. For PBE this rise is within
the scatter. The diagram is decided near the liquidus, which lies between 1728
and 3458 K. So `liquid_temperature` should be as low as possible while every
composition stays liquid. In all runs at 2800 K the energy stayed flat after
the first 5 ps and the mean squared displacement kept growing.

All compositions must run at the same temperature. We tried runs at 50 K above
the mean melting point of the elements of each composition, from 1778 K for Ni
to 3509 K for Re. With GRACE-2L-OMAT this gave L0 of the liquid near
−12 kJ/mol. The potential energy per atom of the liquid rises by 0.171 meV/K
for Ni and by 0.234 meV/K for Re. Over 1778 to 3509 K this difference adds
about −10 kJ/mol to L0.

## Known limitations

- The solid mixing energies are 0 K energies. The vibrational and short-range
  order options of `sqs2tdb` are not used.
- The liquid L0 is the excess energy at `liquid_temperature`. The TDB file uses
  it at all temperatures. L0 changes with temperature, so the liquid has an
  excess heat capacity. Its excess entropy then cannot be zero at all
  temperatures, and the fit leaves it out. It would need free energies, for
  example from thermodynamic integration.
- The Ni side of the diagram is decided near 1800 K, far below 2800 K. A linear
  extrapolation of the mean GRACE-2L-OMAT L0 of three runs gives about
  −1 kJ/mol at 1800 K, against −4.7 kJ/mol at 2800 K.
- An ordered lattice takes the energies of its pure element end members from
  the force field, while FCC_A1 and HCP_A3 take them from SGTE. So pure Re on
  FCC sites can have a lower free energy in NI4MO_D1A than in FCC_A1. This may
  help the ordered phases that all four potentials make stable.
- The mixing energies come from the force fields. A comparison with DFT mixing
  energies of the same SQS would show how much of the difference to experiment
  comes from the force fields.